# CDSDS 542 Sp2026 — Optional LSTM Notebook

This notebook is an **optional extension** for students who want a compact, end-to-end example of sequence modeling with an LSTM.

We will:
- build a small **character-level** dataset,
- train an **LSTM** for **next-character prediction**,
- inspect tensor shapes, and
- compare generated text **before** and **after** training.

This is intentionally lightweight, but it is a complete training example rather than only a shape demo.


In [ ]:
import math
import random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

torch.manual_seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 1. Build a small character dataset

We use a small corpus made from a few short passages repeated multiple times.
This is still tiny compared with real language datasets, but it is large enough to make training more meaningful than a toy string like `"hello"`.


In [ ]:
corpus_parts = [
    "data science is fun when models learn patterns from sequences. ",
    "lstm models process one token at a time and keep a hidden state. ",
    "in this notebook we train a small model for next character prediction. ",
    "the goal is not to build a perfect language model but to understand the pipeline. ",
    "deep learning for data science combines ideas from linear algebra probability and optimization. ",
    "good experiments are simple reproducible and easy to inspect. ",
]

text = "".join(corpus_parts * 25).lower()
print("Corpus length:", len(text))
print(text[:300])


## 2. Vocabulary

At the character level, each unique character gets an integer ID.


In [ ]:
vocab = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)
print(vocab)


## 3. Create training sequences

We use a fixed window of characters as input and ask the model to predict the **next** character at each position.

For example, if the input window is:

`data sc`

then the target window is:

`ata sci`

This is a standard way to frame next-character prediction.


In [ ]:
seq_len = 40

class CharSeqDataset(Dataset):
    def __init__(self, text, stoi, seq_len):
        self.seq_len = seq_len
        self.data = [stoi[ch] for ch in text]

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx:idx + self.seq_len], dtype=torch.long)
        y = torch.tensor(self.data[idx + 1:idx + self.seq_len + 1], dtype=torch.long)
        return x, y

dataset = CharSeqDataset(text, stoi, seq_len)
len(dataset)


In [ ]:
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)

x_batch, y_batch = next(iter(train_loader))
print("x_batch shape:", x_batch.shape)   # (batch, seq_len)
print("y_batch shape:", y_batch.shape)   # (batch, seq_len)


## 4. Define the LSTM model

The model has three parts:
1. an **embedding** layer to turn character IDs into dense vectors,
2. an **LSTM** to process the sequence,
3. a **linear layer** to map hidden states to logits over the vocabulary.


In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        # x: (batch, seq_len)
        emb = self.embedding(x)            # (batch, seq_len, embed_dim)
        out, hidden = self.lstm(emb, hidden)  # out: (batch, seq_len, hidden_dim)
        logits = self.fc(out)              # (batch, seq_len, vocab_size)
        return logits, hidden

model = CharLSTM(vocab_size=vocab_size).to(device)
model


## 5. Shape check

Before training, it is useful to verify the tensor shapes carefully.


In [ ]:
sample_x, sample_y = next(iter(train_loader))
sample_x = sample_x[:4].to(device)

with torch.no_grad():
    logits, hidden = model(sample_x)

print("Input shape: ", sample_x.shape)
print("Logits shape:", logits.shape)
print("h_n shape:   ", hidden[0].shape)
print("c_n shape:   ", hidden[1].shape)


For `nn.CrossEntropyLoss`, we need logits shaped like `(N, C)` and targets shaped like `(N,)`, so we flatten the batch and time dimensions together during training.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_items = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits, _ = model(x)
            loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
            total_loss += loss.item() * x.size(0)
            total_items += x.size(0)
    return total_loss / total_items

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_items = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_items += x.size(0)
    return total_loss / total_items


## 6. Text generation helper

We sample one character at a time from the model’s predicted distribution.


In [ ]:
@torch.no_grad()
def generate_text(model, seed="data ", max_new_chars=200, temperature=1.0):
    model.eval()
    input_ids = [stoi[ch] for ch in seed]
    x = torch.tensor([input_ids], dtype=torch.long, device=device)
    hidden = None

    # warm up the hidden state on the seed
    logits, hidden = model(x, hidden)
    generated = list(seed)
    last_char = x[:, -1:]

    for _ in range(max_new_chars):
        logits, hidden = model(last_char, hidden)
        next_logits = logits[:, -1, :] / temperature
        probs = torch.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        next_char = itos[next_id.item()]
        generated.append(next_char)
        last_char = next_id

    return "".join(generated)

print("Before training:")
print(generate_text(model, seed="data ", max_new_chars=200, temperature=0.8))


## 7. Train the model

Because the dataset is still small, training should run quickly on CPU and even faster on GPU.


In [ ]:
epochs = 12
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch(model, train_loader)
    val_loss = evaluate(model, val_loader)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")


A rough interpretation:
- lower loss means the model is assigning higher probability to the correct next character,
- if both train and validation loss decrease, the model is learning patterns from the corpus.


In [ ]:
print("After training:")
print(generate_text(model, seed="data ", max_new_chars=300, temperature=0.7))


## 8. Optional exercise — understanding shapes

Suppose:
- batch size = 32
- sequence length = 40
- embedding dimension = 32
- hidden dimension = 64
- vocabulary size = 26

Answer:
1. What is the shape after the embedding layer?
2. What is the shape of the LSTM output tensor?
3. What is the shape of the final logits tensor?
4. Why do we reshape logits and targets before `CrossEntropyLoss`?


## 9. Reflection

1. Why is next-character prediction a reasonable first task for understanding sequence models?
2. What does the hidden state of an LSTM help the model remember?
3. Why might an LSTM still struggle compared with modern Transformer-based language models?
4. If we increased the dataset size or model size, what tradeoffs would appear?


## End of optional notebook

This notebook is meant to be a compact, grad-level introduction to **training** an LSTM, while still staying small enough for a discussion section.
